# 04 · LLM / RAG / Agentes — Explicabilidad en Lenguaje Natural
**Integrante 3 — Proyecto Final IA · EAFIT 2026-1**

Este notebook implementa el componente de IA generativa del sistema.  
Dado un estudiante con riesgo de deserción predicho por LightGBM, un LLM  
(vía **Groq API** — gratuita) genera una explicación personalizada en lenguaje natural  
basada en los valores SHAP del modelo.

**Flujo:**
```
Perfil del estudiante
       ↓
LightGBM → probabilidad de deserción
       ↓
SHAP → top features que más influyen en esa predicción
       ↓
Prompt estructurado → Groq (llama-3.1-8b-instant)
       ↓
Explicación en lenguaje natural + recomendaciones
```

**Entradas:**
- `data/processed/X_test.csv` / `y_test.csv`
- `models/checkpoints/lightgbm_dropout_model.pkl`
- `data/processed/feature_names.csv`

**Salidas:**
- Explicaciones generadas para casos de prueba
- `models/llm_eval_results.json` — evaluación cuantitativa (ground truth)

## 0 · Imports y configuración

In [4]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
import sys
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")


def ensure_package(package_name: str) -> None:
    """Instala el paquete si no está disponible en el kernel actual."""
    try:
        __import__(package_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


ensure_package("shap")
ensure_package("python-dotenv")
ensure_package("groq")

import shap
from dotenv import load_dotenv
from groq import Groq

# ── Rutas ──────────────────────────────────────────────────────────────
ROOT       = Path("..")
DATA_PROC  = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
CKPT_DIR   = MODELS_DIR / "checkpoints"

# ── API Key ─────────────────────────────────────────────────────────────
# Opción 1: variable de entorno (recomendada para no hardcodear la key)
#   Windows PowerShell:  $env:GROQ_API_KEY = "gsk_..."
#   Linux/macOS:         export GROQ_API_KEY="gsk_..."
# Opción 2: pegar directamente (no subir al repo)
load_dotenv(ROOT / ".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "PEGA_TU_KEY_AQUI")

client = Groq(api_key=GROQ_API_KEY)
LLM_MODEL = "llama-3.1-8b-instant"   # gratuito en Groq

print(f"Groq client inicializado — modelo: {LLM_MODEL}")
print(f"API key detectada: {'✓' if GROQ_API_KEY != 'PEGA_TU_KEY_AQUI' else '✗ (falta configurar)'}")

Groq client inicializado — modelo: llama-3.1-8b-instant
API key detectada: ✓


## 1 · Carga de datos y modelo

In [5]:
X_test     = pd.read_csv(DATA_PROC / "X_test.csv")
y_test     = pd.read_csv(DATA_PROC / "y_test.csv").squeeze()
feat_names = pd.read_csv(DATA_PROC / "feature_names.csv").iloc[:, 0].tolist()
model      = joblib.load(CKPT_DIR / "lightgbm_dropout_model.pkl")

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Test set cargado: {X_test.shape[0]} muestras, {X_test.shape[1]} features")
print(f"Estudiantes predichos en riesgo: {y_pred.sum()} ({y_pred.mean()*100:.1f}%)")

Test set cargado: 664 muestras, 34 features
Estudiantes predichos en riesgo: 215 (32.4%)


## 2 · Valores SHAP — explicaciones locales

SHAP (SHapley Additive exPlanations) asigna a cada feature una contribución  
al cambio en la predicción respecto a la predicción promedio del modelo.  
Un valor SHAP positivo empuja hacia **mayor riesgo de deserción**;  
un valor negativo empuja hacia **menor riesgo**.

In [6]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# LightGBM con pipeline interno puede devolver lista [clase_0, clase_1]
if isinstance(shap_values, list):
    sv = shap_values[1]   # clase 1 = Deserción
else:
    sv = shap_values

print(f"SHAP values shape: {sv.shape}")
print("SHAP calculados correctamente ✓")

SHAP values shape: (664, 34)
SHAP calculados correctamente ✓


## 3 · Función auxiliar: extraer perfil SHAP de un estudiante

In [7]:
def get_student_shap_profile(idx: int, top_n: int = 6) -> dict:
    """
    Para el estudiante en posición `idx` del test set, retorna:
    - probabilidad de deserción
    - predicción binaria
    - top_n features con mayor |SHAP|, con dirección y valor original
    """
    proba     = float(y_proba[idx])
    predicted = int(y_pred[idx])
    true_lab  = int(y_test.iloc[idx])

    shap_row  = sv[idx]
    top_idx   = np.argsort(np.abs(shap_row))[::-1][:top_n]

    features = []
    for i in top_idx:
        features.append({
            "feature":    feat_names[i],
            "shap_value": round(float(shap_row[i]), 4),
            "direction":  "↑ riesgo" if shap_row[i] > 0 else "↓ riesgo",
            "raw_value":  round(float(X_test.iloc[idx, i]), 3),
        })

    return {
        "student_index":    idx,
        "dropout_prob":     round(proba, 4),
        "predicted_label":  predicted,
        "true_label":       true_lab,
        "correct":          predicted == true_lab,
        "top_features":     features,
    }

# Ejemplo rápido
profile = get_student_shap_profile(0)
print(json.dumps(profile, indent=2, ensure_ascii=False))

{
  "student_index": 0,
  "dropout_prob": 0.0026,
  "predicted_label": 0,
  "true_label": 0,
  "correct": true,
  "top_features": [
    {
      "feature": "Curricular units 2nd sem (approved)",
      "shap_value": -1.3505,
      "direction": "↓ riesgo",
      "raw_value": 0.519
    },
    {
      "feature": "Curricular units 2nd sem (evaluations)",
      "shap_value": -0.5812,
      "direction": "↓ riesgo",
      "raw_value": -0.012
    },
    {
      "feature": "Course",
      "shap_value": -0.5563,
      "direction": "↓ riesgo",
      "raw_value": -1.13
    },
    {
      "feature": "Unemployment rate",
      "shap_value": -0.5072,
      "direction": "↓ riesgo",
      "raw_value": -1.017
    },
    {
      "feature": "Curricular units 1st sem (approved)",
      "shap_value": -0.3993,
      "direction": "↓ riesgo",
      "raw_value": 0.423
    },
    {
      "feature": "Scholarship holder",
      "shap_value": -0.3658,
      "direction": "↓ riesgo",
      "raw_value": 1.776
    }
  ]


## 4 · Diseño del prompt

### Justificación del prompting

Se utiliza **few-shot prompting** con un ejemplo positivo y uno negativo  
para anclar el tono y el formato de la respuesta.  
El system prompt define el rol (orientador académico), el público objetivo  
(consejeros universitarios, no expertos en ML) y las restricciones de formato.  
Las features SHAP se presentan en lenguaje natural con su dirección de efecto  
para que el LLM no necesite interpretar números crudos.

In [8]:
SYSTEM_PROMPT = """Eres un orientador académico de una universidad colombiana.
Tu función es explicar, en lenguaje claro y empático, por qué un estudiante
podría estar en riesgo de deserción, basándote en los factores más influyentes
identificados por un modelo predictivo.

Reglas:
- Dirige la explicación a un consejero universitario (no a un experto en ML).
- Menciona los 3 factores más importantes por nombre, explica su efecto y da
  una recomendación concreta para cada uno.
- Tono: profesional, empático, nunca alarmista.
- Longitud: máximo 180 palabras.
- No inventes datos que no estén en el perfil.
- Responde siempre en español."""

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": """Perfil del estudiante:
- Probabilidad de deserción: 0.82
- Factores de riesgo principales:
  1. Materias aprobadas 2do semestre: ↑ riesgo (aprobó muy pocas)
  2. Al día con pagos de matrícula: ↑ riesgo (no está al día)
  3. Nota promedio 1er semestre: ↑ riesgo (nota muy baja)"""
    },
    {
        "role": "assistant",
        "content": """Este estudiante presenta un riesgo alto de deserción (82%). Los tres factores más críticos son:

1. **Bajo rendimiento académico**: aprobó muy pocas materias en el segundo semestre y obtuvo una nota promedio baja en el primero. Se recomienda activar un plan de acompañamiento académico y revisar si hay dificultades con materias específicas.

2. **Situación financiera**: no está al día con los pagos de matrícula, lo que puede generar bloqueos administrativos. Es prioritario orientarlo hacia las oficinas de financiamiento y becas disponibles.

3. **Acumulación de factores**: la combinación de bajo rendimiento y dificultades financieras amplifica el riesgo. Se sugiere una cita con el consejero de bienestar universitario esta semana."""
    },
    {
        "role": "user",
        "content": """Perfil del estudiante:
- Probabilidad de deserción: 0.21
- Factores protectores principales:
  1. Materias aprobadas 2do semestre: ↓ riesgo (aprobó la mayoría)
  2. Beca activa: ↓ riesgo (tiene beca)
  3. Nota promedio 2do semestre: ↓ riesgo (nota alta)"""
    },
    {
        "role": "assistant",
        "content": """Este estudiante muestra un riesgo bajo de deserción (21%). Los factores protectores más relevantes son:

1. **Buen rendimiento académico**: aprobó la mayoría de materias con notas altas en el segundo semestre, lo que indica adaptación y compromiso con el programa.

2. **Estabilidad financiera**: cuenta con beca activa, lo que reduce la presión económica y le permite enfocarse en sus estudios.

3. **Trayectoria positiva**: la combinación de rendimiento y apoyo financiero sitúa a este estudiante en una posición favorable. Se recomienda mantener el seguimiento periódico para detectar cambios tempranos."""
    },
]

print("System prompt y few-shot examples definidos ✓")
print(f"\nSystem prompt ({len(SYSTEM_PROMPT)} chars):")
print(SYSTEM_PROMPT[:200] + "...")

System prompt y few-shot examples definidos ✓

System prompt (621 chars):
Eres un orientador académico de una universidad colombiana.
Tu función es explicar, en lenguaje claro y empático, por qué un estudiante
podría estar en riesgo de deserción, basándote en los factores m...


## 5 · Función de explicación vía Groq

In [9]:
def build_user_message(profile: dict) -> str:
    """Construye el mensaje de usuario a partir del perfil SHAP."""
    prob   = profile["dropout_prob"]
    feats  = profile["top_features"][:3]   # top 3 para el prompt
    nivel  = "alto" if prob >= 0.6 else ("moderado" if prob >= 0.4 else "bajo")

    lines = [
        f"Perfil del estudiante:",
        f"- Probabilidad de deserción: {prob} (riesgo {nivel})",
        f"- Factores principales:",
    ]
    for i, f in enumerate(feats, 1):
        lines.append(
            f"  {i}. {f['feature']}: {f['direction']} "
            f"(valor estandarizado: {f['raw_value']})"
        )
    return "\n".join(lines)


def explain_student(idx: int, top_n: int = 6) -> dict:
    """
    Genera una explicación en lenguaje natural para el estudiante `idx`.
    Retorna el perfil SHAP + la explicación del LLM + metadata de la llamada.
    """
    profile      = get_student_shap_profile(idx, top_n=top_n)
    user_message = build_user_message(profile)

    messages = FEW_SHOT_EXAMPLES + [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages,
        temperature=0.4,   # bajo para reproducibilidad
        max_tokens=300,
    )

    explanation = response.choices[0].message.content.strip()
    tokens_used = response.usage.total_tokens

    return {
        **profile,
        "user_message":  user_message,
        "explanation":   explanation,
        "tokens_used":   tokens_used,
        "llm_model":     LLM_MODEL,
        "temperature":   0.4,
    }

print("Función explain_student definida ✓")

Función explain_student definida ✓


## 6 · Demo — explicaciones para casos representativos

Se seleccionan 4 casos del test set:
- **Caso A:** alto riesgo, predicción correcta (TP)
- **Caso B:** bajo riesgo, predicción correcta (TN)
- **Caso C:** falso negativo (FN) — estudiante en riesgo no detectado
- **Caso D:** falso positivo (FP) — falsa alarma

In [10]:
import numpy as np

# Índices representativos
tp_idx = np.where((y_pred == 1) & (y_test.values == 1))[0]
tn_idx = np.where((y_pred == 0) & (y_test.values == 0))[0]
fn_idx = np.where((y_pred == 0) & (y_test.values == 1))[0]
fp_idx = np.where((y_pred == 1) & (y_test.values == 0))[0]

# Elegir el caso con mayor probabilidad para TP, menor para TN
caso_A = tp_idx[np.argmax(y_proba[tp_idx])]
caso_B = tn_idx[np.argmin(y_proba[tn_idx])]
caso_C = fn_idx[np.argmax(y_proba[fn_idx])]   # FN con mayor proba (frontera)
caso_D = fp_idx[np.argmax(y_proba[fp_idx])]   # FP con mayor proba (más confiado)

casos = {
    "A — Alto riesgo (TP)":         caso_A,
    "B — Bajo riesgo (TN)":         caso_B,
    "C — Falso Negativo (FN)":      caso_C,
    "D — Falso Positivo (FP)":      caso_D,
}
print("Casos seleccionados:")
for nombre, idx in casos.items():
    print(f"  {nombre}: idx={idx}, proba={y_proba[idx]:.3f}, "
          f"pred={y_pred[idx]}, true={y_test.iloc[idx]}")

Casos seleccionados:
  A — Alto riesgo (TP): idx=278, proba=1.000, pred=1, true=1
  B — Bajo riesgo (TN): idx=134, proba=0.000, pred=0, true=0
  C — Falso Negativo (FN): idx=90, proba=0.497, pred=0, true=1
  D — Falso Positivo (FP): idx=382, proba=0.997, pred=1, true=0


In [11]:
resultados_demo = {}

for nombre, idx in casos.items():
    print(f"\n{'='*60}")
    print(f"  {nombre}")
    print('='*60)
    resultado = explain_student(idx)
    resultados_demo[nombre] = resultado
    print(f"\n📋 Prompt enviado:\n{resultado['user_message']}")
    print(f"\n🤖 Explicación LLM ({resultado['tokens_used']} tokens):")
    print(resultado['explanation'])


  A — Alto riesgo (TP)

📋 Prompt enviado:
Perfil del estudiante:
- Probabilidad de deserción: 1.0 (riesgo alto)
- Factores principales:
  1. Tuition fees up to date: ↑ riesgo (valor estandarizado: -2.744)
  2. Curricular units 2nd sem (approved): ↑ riesgo (valor estandarizado: -1.482)
  3. Age at enrollment: ↑ riesgo (valor estandarizado: 3.401)

🤖 Explicación LLM (1085 tokens):
Este estudiante presenta un riesgo extremadamente alto de deserción (probabilidad de 1.0). Los factores más críticos son:

1. **Dificultades financieras**: no está al día con los pagos de matrícula, lo que puede generar bloqueos administrativos y estrés adicional.

2. **Bajo rendimiento académico**: aprobó muy pocas materias en el segundo semestre, lo que indica dificultades para adaptarse al programa.

3. **Edad avanzada**: es un estudiante adulto que se inscribió en la universidad, lo que puede ser un desafío para su adaptación y compromiso con el programa. Es posible que esté enfrentando responsabilidades f

## 7 · Evaluación cuantitativa del componente LLM

Se evalúa la calidad de las explicaciones con un **mini ground truth** de 8 casos,  
verificando 3 criterios binarios por explicación:

| Criterio | Descripción |
|---|---|
| `menciona_factores_correctos` | El LLM menciona al menos 2 de los 3 top features SHAP |
| `tono_adecuado` | La explicación es empática y no alarmista |
| `da_recomendacion` | Incluye al menos una recomendación concreta |

La evaluación se hace con un segundo LLM call como *juez* (LLM-as-a-judge).

In [12]:
JUDGE_PROMPT = """Eres un evaluador de calidad de explicaciones académicas.
Dada una explicación generada por IA sobre riesgo de deserción estudiantil,
evalúa los siguientes criterios. Responde SOLO con un JSON válido, sin texto adicional.

Criterios:
1. menciona_factores_correctos (true/false): ¿La explicación menciona al menos 2
   de los factores del perfil por nombre o concepto equivalente?
2. tono_adecuado (true/false): ¿El tono es empático, profesional y no alarmista?
3. da_recomendacion (true/false): ¿Incluye al menos una recomendación concreta?

Formato de respuesta (exacto):
{
  "menciona_factores_correctos": true,
  "tono_adecuado": true,
  "da_recomendacion": false,
  "justificacion": "frase breve"
}"""

def evaluate_explanation(profile: dict, explanation: str) -> dict:
    """Usa el LLM como juez para evaluar la calidad de la explicación."""
    eval_prompt = f"""Perfil enviado al sistema:
{profile['user_message']}

Explicación generada:
{explanation}

Evalúa según los criterios indicados."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user",   "content": eval_prompt},
        ],
        temperature=0.0,
        max_tokens=200,
    )
    raw = response.choices[0].message.content.strip()
    # Limpiar posibles ```json ... ```
    raw = raw.replace("```json","").replace("```","").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"error": "parse_failed", "raw": raw}

print("Función evaluate_explanation definida ✓")

Función evaluate_explanation definida ✓


In [13]:
# Evaluar 8 casos: los 4 del demo + 4 adicionales aleatorios
np.random.seed(42)
extra_idx = np.random.choice(len(X_test), 4, replace=False).tolist()
eval_cases = list(casos.values()) + extra_idx

eval_results = []
print("Evaluando 8 casos...\n")

for i, idx in enumerate(eval_cases):
    resultado  = explain_student(int(idx))
    evaluacion = evaluate_explanation(resultado, resultado["explanation"])

    eval_results.append({
        "case_idx":  int(idx),
        "dropout_prob": resultado["dropout_prob"],
        "predicted":    resultado["predicted_label"],
        "true_label":   resultado["true_label"],
        **{k: v for k, v in evaluacion.items() if k != "justificacion"},
        "justificacion": evaluacion.get("justificacion",""),
    })

    criterios = [evaluacion.get("menciona_factores_correctos","?"),
                 evaluacion.get("tono_adecuado","?"),
                 evaluacion.get("da_recomendacion","?")]
    score = sum(1 for c in criterios if c is True)
    print(f"Caso {i+1} (idx={idx}, p={resultado['dropout_prob']:.2f}): "
          f"score={score}/3 | {evaluacion.get('justificacion','')[:60]}")

# Guardar resultados
with open(MODELS_DIR / "llm_eval_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)
print("\nGuardado: models/llm_eval_results.json")

Evaluando 8 casos...

Caso 1 (idx=278, p=1.00): score=3/3 | La explicación incluye una recomendación concreta para un en
Caso 2 (idx=134, p=0.00): score=2/3 | No se incluye ninguna recomendación concreta para el seguimi
Caso 3 (idx=90, p=0.50): score=3/3 | Se incluyen recomendaciones concretas para cada factor, como
Caso 4 (idx=382, p=1.00): score=3/3 | La explicación incluye una recomendación concreta para una i
Caso 5 (idx=281, p=1.00): score=3/3 | Se incluye una recomendación concreta para una evaluación ex
Caso 6 (idx=286, p=0.86): score=3/3 | La explicación incluye recomendaciones concretas para cada f
Caso 7 (idx=473, p=0.11): score=3/3 | Se recomienda ofrecer apoyo emocional y orientación para man
Caso 8 (idx=227, p=1.00): score=3/3 | Incluye recomendaciones concretas de apoyo financiero y acad

Guardado: models/llm_eval_results.json


In [14]:
# Resumen estadístico
df_eval = pd.DataFrame(eval_results)
criterios_cols = ["menciona_factores_correctos", "tono_adecuado", "da_recomendacion"]

print("\n╔══════════════════════════════════════════════╗")
print("║     EVALUACIÓN CUANTITATIVA — LLM (n=8)      ║")
print("╠══════════════════════════════════════════════╣")
for col in criterios_cols:
    if col in df_eval.columns:
        tasa = df_eval[col].mean() * 100
        print(f"║  {col:<35} {tasa:5.1f}% ║")
print("╠══════════════════════════════════════════════╣")
score_total = df_eval[criterios_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1).mean()
print(f"║  Score promedio global:              {score_total*100:5.1f}% ║")
print("╚══════════════════════════════════════════════╝")


╔══════════════════════════════════════════════╗
║     EVALUACIÓN CUANTITATIVA — LLM (n=8)      ║
╠══════════════════════════════════════════════╣
║  menciona_factores_correctos         100.0% ║
║  tono_adecuado                       100.0% ║
║  da_recomendacion                     87.5% ║
╠══════════════════════════════════════════════╣
║  Score promedio global:               95.8% ║
╚══════════════════════════════════════════════╝


## 8 · Resumen del componente LLM

### Decisiones de diseño documentadas

| Decisión | Elección | Justificación |
|---|---|---|
| **Modelo LLM** | `llama-3.1-8b-instant` vía Groq | API gratuita, baja latencia, suficiente para generación en español |
| **Estrategia de prompting** | Few-shot (2 ejemplos) | Ancla el formato y tono sin necesidad de fine-tuning |
| **Temperatura** | 0.4 | Balance entre reproducibilidad y naturalidad del texto |
| **Fuente de features** | SHAP TreeExplainer | Explicaciones locales fieles al modelo LightGBM |
| **Evaluación** | LLM-as-a-judge (3 criterios binarios) | Permite evaluación cuantitativa sin anotadores humanos |

### Rol del componente en el sistema

El módulo LLM **no reemplaza** al modelo predictivo — lo complementa.  
LightGBM produce la probabilidad de riesgo; SHAP identifica qué variables  
la explican; el LLM traduce esos factores técnicos a lenguaje natural accionable  
para consejeros universitarios que no tienen formación en ML.